In [ ]:
import os
import pickle
from tqdm import tqdm
import numpy as np
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model
print("START")

base_model = InceptionV3(weights='imagenet')
model = Model(
    inputs=base_model.input,
    outputs=base_model.layers[-2].output
)
def extract_features():
    directory = r"C:\\Users\\komal\\Documents\\pbl\\archive\\Images"
    features = {}
    for img_name in tqdm(os.listdir(directory)):
        img_path = os.path.join(directory, img_name)
        try:
            image = load_img(img_path, target_size=(299, 299))
            image = img_to_array(image)
            image = np.expand_dims(image, axis=0)
            image = preprocess_input(image)
            feature = model.predict(image, verbose=0)
            image_id = img_name.split('.')[0]
            features[image_id] = feature.flatten()
        except Exception as e:
            print(f"Error processing {img_name}: {e}")

    with open("features.pkl", "wb") as f:
        pickle.dump(features, f)

    print(f"Total images processed: {len(features)}")

if __name__ == "__main__":
    extract_features()

In [ ]:
import pickle
import string
from tensorflow.keras.preprocessing.text import Tokenizer

def clean_text():
    with open("C:\\Users\\komal\\Documents\\pbl\\archive\\captions.txt", "r", encoding="utf-8") as f:
        next(f)
        lines = f.readlines()
    mapping = {}
    for line in lines:
        tokens = line.strip().split(",", 1)
        if len(tokens) < 2:
            continue
        img_name, caption = tokens[0], tokens[1]
        img_id = img_name.split(".")[0]
        caption = caption.lower().strip()
        caption = caption.translate(str.maketrans("", "", string.punctuation))
        caption = " ".join([word for word in caption.split() if len(word) > 1])
        caption = "startseq " + caption + " endseq"
        if img_id not in mapping:
            mapping[img_id] = []
        mapping[img_id].append(caption)
    all_captions = [cap for caps in mapping.values() for cap in caps]

    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(all_captions)
    with open("tokenizer.pkl", "wb") as f:
        pickle.dump(tokenizer, f)
        
    with open("captions_clean.pkl", "wb") as f:
        pickle.dump(mapping, f)

    print("Tokenizer saved as tokenizer.pkl")
    print("Clean captions saved as captions_clean.pkl")
    print("Total images:", len(mapping))
    print("Total captions:", len(all_captions))


if __name__ == "__main__":
    clean_text()

In [ ]:
import os
import pickle
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau


# 1. Load Data
print("Loading components for the high-performance run")
features = pickle.load(open("features.pkl", "rb"))
tokenizer = pickle.load(open("tokenizer.pkl", "rb"))
vocab_size = len(tokenizer.word_index) + 1
max_length = 36   # Matches main.py

# 2. Setup Mapping
mapping = {}
with open("data/captions.txt", "r") as f:
    next(f)
    for line in f:
        tokens = line.split(",")
        if len(tokens) < 2:
            continue
        img_id, caption = tokens[0].split(".")[0], tokens[1].lower().strip()
        if img_id not in mapping:
            mapping[img_id] = []
        mapping[img_id].append(f"startseq {caption} endseq")

In [ ]:
import pyttsx3

def speak_caption(caption):
    """
    Takes a string (caption) and reads it out loud.
    """
    engine = pyttsx3.init()
    # Cleaning the caption for a natural voice
    clean_text = caption.replace("startseq", "").replace("endseq", "").strip()
    print(f"VisionVoice is saying: {clean_text}")
    engine.setProperty("rate", 150)
    engine.say(clean_text)
    engine.runAndWait()


if __name__ == "__main__":
    # Test run
    sample_text = "startseq a young boy is playing with a red ball in the park endseq"
    speak_caption(sample_text)